In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='projectmoviesscope')]

In [0]:
spark.conf.set(
  "fs.azure.account.key.moviessaw.dfs.core.windows.net",
  dbutils.secrets.get(scope="projectmoviesscope", key="storagAccountKey")
)

In [0]:
dbutils.fs.ls("abfss://projectmoviesdata@moviessaw.dfs.core.windows.net/")

[FileInfo(path='abfss://projectmoviesdata@moviessaw.dfs.core.windows.net/Silver/', name='Silver/', size=0, modificationTime=1777742402000),
 FileInfo(path='abfss://projectmoviesdata@moviessaw.dfs.core.windows.net/raw-data/', name='raw-data/', size=0, modificationTime=1774605205000),
 FileInfo(path='abfss://projectmoviesdata@moviessaw.dfs.core.windows.net/transformed-data/', name='transformed-data/', size=0, modificationTime=1774614286000),
 FileInfo(path='abfss://projectmoviesdata@moviessaw.dfs.core.windows.net/transformed_data/', name='transformed_data/', size=0, modificationTime=1774603645000)]

In [0]:
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .load("abfss://datalake@moviessaw.dfs.core.windows.net/Bronze/employee1.csv")
display(employee_df)


Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
Bachelors,2017,Bangalore,3,34,Male,No,0,0
Bachelors,2013,Pune,1,28,Female,No,3,1
Bachelors,2014,New Delhi,3,38,Female,No,2,0
Masters,2016,Bangalore,3,27,Male,No,5,1
Masters,2017,Pune,3,24,Male,Yes,2,1
Bachelors,2016,Bangalore,3,22,Male,No,0,0
Bachelors,2015,New Delhi,3,38,Male,No,0,0
Bachelors,2016,Bangalore,3,34,Female,No,2,1
Bachelors,2016,Pune,3,23,Male,No,1,0
Masters,2017,New Delhi,2,37,Male,No,2,0


In [0]:
sales_df = spark.read.format("csv") \
    .option("header", "true") \
    .load("abfss://datalake@moviessaw.dfs.core.windows.net/Bronze/salesdata1.csv")
display(sales_df)

Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.4,Returning,0.2,Bank Transfer,Retail,South-David
1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
1061,2023-03-24,Charlie,East,3750.2,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie
1021,2023-02-11,Charlie,West,3761.15,32,Food,900.79,1106.51,New,0.21,Cash,Online,West-Charlie
1083,2023-04-11,Bob,West,618.31,29,Furniture,2408.81,2624.09,Returning,0.14,Cash,Online,West-Bob
1087,2023-01-06,Eve,South,7698.92,46,Furniture,3702.51,3964.65,New,0.12,Bank Transfer,Online,South-Eve
1075,2023-06-29,David,South,4223.39,30,Furniture,738.06,1095.4499999999998,New,0.05,Bank Transfer,Online,South-David
1075,2023-10-09,Charlie,West,8239.58,18,Clothing,2228.35,2682.34,New,0.13,Bank Transfer,Online,West-Charlie


In [0]:
employee_df.printSchema()


root
 |-- Education: string (nullable = true)
 |-- JoiningYear: string (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: string (nullable = true)
 |-- LeaveOrNot: string (nullable = true)



In [0]:
sales_df.printSchema()


root
 |-- Product_ID: string (nullable = true)
 |-- Sale_Date: string (nullable = true)
 |-- Sales_Rep: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales_Amount: string (nullable = true)
 |-- Quantity_Sold: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Unit_Cost: string (nullable = true)
 |-- Unit_Price: string (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Region_and_Sales_Rep: string (nullable = true)



In [0]:
print("Employee rows:", employee_df.count(), "columns:", len(employee_df.columns))
print("Sales rows:", sales_df.count(), "columns:", len(sales_df.columns))

Employee rows: 4653 columns: 9
Sales rows: 1000 columns: 14


In [0]:
from pyspark.sql.functions import col, sum

employee_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in employee_df.columns]).show()

sales_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in sales_df.columns]).show()

+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+
|        0|          0|   0|          0|  0|     0|          0|                        0|         0|
+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+

+----------+---------+---------+------+------------+-------------+----------------+---------+----------+-------------+--------+--------------+-------------+--------------------+
|Product_ID|Sale_Date|Sales_Rep|Region|Sales_Amount|Quantity_Sold|Product_Category|Unit_Cost|Unit_Price|Customer_Type|Discount|Payment_Method|Sales_Channel|Region_and_Sales_Rep|
+----------+---------+---------+------+------------+-------------+----------------+---------+----------+-------------+--------+-----------

In [0]:
print("Employee duplicates:", employee_df.count() - employee_df.dropDuplicates().count())
print("Sales duplicates:", sales_df.count() - sales_df.dropDuplicates().count())

Employee duplicates: 1889
Sales duplicates: 0


In [0]:
employee_df.describe().display()

summary,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
count,4653,4653,4653,4653,4653,4653,4653,4653,4653
mean,null,2015.0629701267999,null,2.6982591876208897,29.393294648613796,null,null,2.905652267354395,0.3438641736514077
stddev,null,1.863376828686355,null,0.5614354643364909,4.826087009126064,null,null,1.5582403309268569,0.47504747514881024
min,Bachelors,2012,Bangalore,1,22,Female,No,0,0
max,PHD,2018,Pune,3,41,Male,Yes,7,1


In [0]:
sales_df.describe().display()

summary,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
count,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
mean,1050.128,null,null,null,5019.265230000003,25.355,null,2475.30455,2728.440120000001,null,0.15239000000000005,null,null,null
stddev,29.573505172843596,null,null,null,2846.7901256682326,14.159006054538308,null,1417.8725460040862,1419.3998385807104,null,0.08719972300237924,null,null,null
min,1001,2023-01-01,Alice,East,100.12,1,Clothing,1000.18,1001.77,New,0.0,Bank Transfer,Online,East-Alice
max,1100,2024-01-01,Eve,West,9989.04,9,Furniture,999.09,999.1800000000001,Returning,0.3,Credit Card,Retail,West-Eve


In [0]:
employee_df.select("Education").distinct().show()
sales_df.select("Region").distinct().show()

+---------+
|Education|
+---------+
|  Masters|
|Bachelors|
|      PHD|
+---------+

+------+
|Region|
+------+
| South|
|  East|
|  West|
| North|
+------+



In [0]:
display(employee_df.groupBy("Age", "LeaveOrNot").count())

Age,LeaveOrNot,count
26,0,422
31,1,37
26,1,223
34,0,94
37,0,98
36,1,45
29,0,155
39,1,39
32,0,78
33,1,40


Databricks visualization. Run in Databricks to view.

In [0]:
display(employee_df.groupBy("Gender", "LeaveOrNot").count())

Gender,LeaveOrNot,count
Male,1,716
Female,0,991
Male,0,2062
Female,1,884


Databricks visualization. Run in Databricks to view.

In [0]:
display(employee_df.groupBy("Education", "LeaveOrNot").count())

Education,LeaveOrNot,count
Masters,0,447
Bachelors,0,2472
PHD,1,45
PHD,0,134
Bachelors,1,1129
Masters,1,426


Databricks visualization. Run in Databricks to view.

In [0]:
display(employee_df.groupBy("ExperienceInCurrentDomain", "LeaveOrNot").count())

ExperienceInCurrentDomain,LeaveOrNot,count
7,1,3
4,1,297
5,1,288
0,1,124
1,1,188
7,0,6
2,0,688
0,0,231
5,0,631
3,1,299


Databricks visualization. Run in Databricks to view.

In [0]:
display(employee_df.groupBy("Education", "Gender", "LeaveOrNot").count())

Education,Gender,LeaveOrNot,count
Bachelors,Female,1,698
PHD,Male,0,83
PHD,Female,1,18
PHD,Female,0,51
Bachelors,Male,1,431
Masters,Male,1,258
Masters,Female,1,168
PHD,Male,1,27
Masters,Male,0,244
Bachelors,Male,0,1735


In [0]:
display(sales_df)

Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.4,Returning,0.2,Bank Transfer,Retail,South-David
1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
1061,2023-03-24,Charlie,East,3750.2,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie
1021,2023-02-11,Charlie,West,3761.15,32,Food,900.79,1106.51,New,0.21,Cash,Online,West-Charlie
1083,2023-04-11,Bob,West,618.31,29,Furniture,2408.81,2624.09,Returning,0.14,Cash,Online,West-Bob
1087,2023-01-06,Eve,South,7698.92,46,Furniture,3702.51,3964.65,New,0.12,Bank Transfer,Online,South-Eve
1075,2023-06-29,David,South,4223.39,30,Furniture,738.06,1095.4499999999998,New,0.05,Bank Transfer,Online,South-David
1075,2023-10-09,Charlie,West,8239.58,18,Clothing,2228.35,2682.34,New,0.13,Bank Transfer,Online,West-Charlie


In [0]:
from pyspark.sql.functions import col, round

sales_df = sales_df.withColumn("Sales_Amount", round(col("Sales_Amount").cast("double"), 2))

In [0]:
from pyspark.sql.functions import sum, round

display(
    sales_df.groupBy("Region")
    .agg(round(sum("Sales_Amount"), 2).alias("Total_Sales")))

Region,Total_Sales
South,1154250.86
East,1259792.93
West,1235608.93
North,1369612.51


Databricks visualization. Run in Databricks to view.

In [0]:
display(sales_df.groupBy("Product_Category").sum("Sales_Amount"))

Product_Category,sum(Sales_Amount)
Food,1201773.5399999998
Electronics,1243499.6400000006
Clothing,1313474.3599999999
Furniture,1260517.6900000002


Databricks visualization. Run in Databricks to view.

In [0]:
display(sales_df.groupBy("Sales_Channel").sum("Sales_Amount"))

Sales_Channel,sum(Sales_Amount)
Online,2458833.9300000034
Retail,2560431.2999999984


Databricks visualization. Run in Databricks to view.

In [0]:
display(sales_df.groupBy("Customer_Type").sum("Sales_Amount"))

Customer_Type,sum(Sales_Amount)
Returning,2513006.930000002
New,2506258.299999998


Databricks visualization. Run in Databricks to view.

In [0]:
display(sales_df.select("Quantity_Sold", "Sales_Amount"))

Quantity_Sold,Sales_Amount
18,5053.97
17,4384.02
30,4631.23
39,2167.94
13,3750.2
32,3761.15
29,618.31
46,7698.92
30,4223.39
18,8239.58


In [0]:
display(sales_df.groupBy("Region", "Product_Category").sum("Sales_Amount"))

Region,Product_Category,sum(Sales_Amount)
South,Clothing,269517.74
North,Clothing,372977.22
North,Electronics,342666.29000000015
East,Electronics,303101.42
East,Food,325864.87
East,Clothing,356670.4000000001
West,Clothing,314309.00000000006
South,Furniture,289881.6500000001
South,Food,301187.51
West,Electronics,304067.97000000003


Databricks visualization. Run in Databricks to view.

In [0]:
display(employee_df.head())

Row(Education='Bachelors', JoiningYear='2017', City='Bangalore', PaymentTier='3', Age='34', Gender='Male', EverBenched='No', ExperienceInCurrentDomain='0', LeaveOrNot='0')

In [0]:
from pyspark.sql.functions import col, when

# Remove duplicates
employee_silver = employee_df.dropDuplicates()

# Remove null values
employee_silver = employee_silver.dropna()

# Convert 'EverBenched' from 'Yes'/'No' to 1/0
employee_silver = employee_silver.withColumn(
    "EverBenched",
    when(col("EverBenched") == "Yes", 1)
    .when(col("EverBenched") == "No", 0)
    .otherwise(None)
)

# Cast columns to correct data types
employee_silver = employee_silver \
    .withColumn("JoiningYear", col("JoiningYear").cast("int")) \
    .withColumn("PaymentTier", col("PaymentTier").cast("int")) \
    .withColumn("Age", col("Age").cast("int")) \
    .withColumn("EverBenched", col("EverBenched").cast("int")) \
    .withColumn("ExperienceInCurrentDomain", col("ExperienceInCurrentDomain").cast("int")) \
    .withColumn("LeaveOrNot", col("LeaveOrNot").cast("int"))

# Display cleaned data
display(employee_silver)

Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
Bachelors,2016,Pune,3,26,Male,0,4,0
Bachelors,2014,Bangalore,3,28,Male,0,3,0
Masters,2015,New Delhi,3,28,Male,0,2,0
Bachelors,2014,New Delhi,3,26,Male,0,4,0
Masters,2017,Pune,2,30,Female,0,2,1
Bachelors,2016,New Delhi,3,27,Female,0,5,1
Bachelors,2017,New Delhi,2,29,Male,0,3,1
Bachelors,2017,Pune,3,30,Male,0,0,0
Bachelors,2017,New Delhi,2,39,Female,0,0,0
Bachelors,2014,Bangalore,3,39,Male,0,5,1


In [0]:
employee_silver.printSchema()

root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: integer (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, when, to_date

# Remove duplicates
sales_silver = sales_df.dropDuplicates()

# Remove null values
sales_silver = sales_silver.dropna()

# Cast columns to correct data types
sales_silver = sales_silver \
    .withColumn("Product_ID", col("Product_ID").cast("int")) \
    .withColumn("Sale_Date", to_date(col("Sale_Date"), "yyyy-MM-dd")) \
    .withColumn("Sales_Amount", col("Sales_Amount").cast("double")) \
    .withColumn("Quantity_Sold", col("Quantity_Sold").cast("int")) \
    .withColumn("Unit_Cost", col("Unit_Cost").cast("double")) \
    .withColumn("Unit_Price", col("Unit_Price").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double"))

# Display cleaned data
display(sales_silver)

Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
1003,2023-09-14,Bob,West,3755.88,15,Clothing,3286.24,3388.8399999999997,Returning,0.09,Credit Card,Retail,West-Bob
1036,2023-10-20,David,North,6499.94,49,Clothing,1247.1,1429.4399999999998,New,0.16,Cash,Online,North-David
1090,2023-06-28,Charlie,North,5976.25,41,Clothing,3736.88,3946.77,Returning,0.06,Credit Card,Online,North-Charlie
1058,2023-06-20,Eve,West,2896.54,48,Clothing,2614.48,3049.04,New,0.1,Credit Card,Retail,West-Eve
1096,2023-06-10,Charlie,South,2320.51,6,Furniture,252.62,742.0,New,0.22,Credit Card,Retail,South-Charlie
1032,2023-04-30,Alice,West,4241.51,18,Electronics,2466.18,2483.87,Returning,0.08,Cash,Online,West-Alice
1063,2023-08-15,Alice,East,2780.98,26,Food,1046.96,1528.15,New,0.07,Credit Card,Retail,East-Alice
1051,2023-11-16,Bob,West,803.25,31,Furniture,144.88,175.29,New,0.07,Bank Transfer,Online,West-Bob
1093,2023-09-18,David,West,4040.25,19,Electronics,3808.59,3844.51,New,0.21,Credit Card,Online,West-David
1082,2023-11-30,Alice,South,6261.9,41,Electronics,1196.42,1444.97,Returning,0.21,Bank Transfer,Online,South-Alice


In [0]:
sales_silver.printSchema()

root
 |-- Product_ID: integer (nullable = true)
 |-- Sale_Date: date (nullable = true)
 |-- Sales_Rep: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales_Amount: double (nullable = true)
 |-- Quantity_Sold: integer (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Unit_Cost: double (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Region_and_Sales_Rep: string (nullable = true)



In [0]:
sales_silver.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Silver/sales/")

employee_silver.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Silver/employee/")

In [0]:
sales_silver = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Silver/sales/")

employee_silver = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Silver/employee/")

In [0]:
from pyspark.sql.functions import col

employee_silver = employee_silver \
    .withColumn("Age", col("Age").cast("int")) \
    .withColumn("JoiningYear", col("JoiningYear").cast("int")) \
    .withColumn("PaymentTier", col("PaymentTier").cast("int")) \
    .withColumn("ExperienceInCurrentDomain", col("ExperienceInCurrentDomain").cast("int")) \
    .withColumn("LeaveOrNot", col("LeaveOrNot").cast("int"))


from pyspark.sql.functions import to_date

sales_silver = sales_silver \
    .withColumn("Sales_Amount", col("Sales_Amount").cast("double")) \
    .withColumn("Quantity_Sold", col("Quantity_Sold").cast("int")) \
    .withColumn("Unit_Cost", col("Unit_Cost").cast("double")) \
    .withColumn("Unit_Price", col("Unit_Price").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double")) \
    .withColumn("Sale_Date", to_date(col("Sale_Date"), "yyyy-MM-dd"))

In [0]:
from pyspark.sql.functions import avg, round

gold_experience_attrition_rate = employee_silver.groupBy("ExperienceInCurrentDomain") \
    .agg(round(avg("LeaveOrNot"), 2).alias("Attrition_Rate")) \
    .orderBy("ExperienceInCurrentDomain")

display(gold_experience_attrition_rate)

ExperienceInCurrentDomain,Attrition_Rate
0,0.38
1,0.37
2,0.43
3,0.43
4,0.39
5,0.34
6,0.25
7,0.33


In [0]:
gold_age_attrition_rate = employee_silver.groupBy("Age") \
    .agg(round(avg("LeaveOrNot"), 2).alias("Attrition_Rate")) \
    .orderBy("Age")

display(gold_age_attrition_rate)

Age,Attrition_Rate
22,0.45
23,0.32
24,0.5
25,0.53
26,0.53
27,0.47
28,0.39
29,0.36
30,0.4
31,0.3


In [0]:
gold_discount_effect = sales_silver.groupBy("Discount") \
    .agg(round(avg("Sales_Amount"), 2).alias("Avg_Sales")) \
    .orderBy("Discount")

display(gold_discount_effect)

Discount,Avg_Sales
0.0,4511.5
0.01,5103.34
0.02,4435.17
0.03,5604.86
0.04,5234.55
0.05,5741.82
0.06,4432.35
0.07,4599.94
0.08,4967.66
0.09,4794.34


In [0]:
from pyspark.sql.functions import sum

gold_quantity_revenue = sales_silver.groupBy("Quantity_Sold") \
    .agg(round(sum("Sales_Amount"), 2).alias("Total_Revenue")) \
    .orderBy("Quantity_Sold")

display(gold_quantity_revenue)

Quantity_Sold,Total_Revenue
1,96007.87
2,108691.16
3,73111.32
4,114444.85
5,69266.69
6,94913.11
7,90102.79
8,86281.12
9,84207.78
10,87890.39


In [0]:

# Employee Gold
gold_experience_attrition_rate.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/employee/gold_experience_attrition_rate/")

gold_age_attrition_rate.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/employee/gold_age_attrition_rate/")

# Sales Gold
gold_discount_effect.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/sales/gold_discount_effect/")

gold_quantity_revenue.write.mode("overwrite") \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/sales/gold_quantity_revenue/")

In [0]:

gold_experience_attrition_rate = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/employee/gold_experience_attrition_rate/")

gold_age_attrition_rate = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/employee/gold_age_attrition_rate/")

gold_discount_effect = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/sales/gold_discount_effect/")

gold_quantity_revenue = spark.read \
    .option("header", True) \
    .csv("abfss://datalake@moviessaw.dfs.core.windows.net/Gold/sales/gold_quantity_revenue/")

In [0]:
display(gold_experience_attrition_rate)

ExperienceInCurrentDomain,Attrition_Rate
0,0.38
1,0.37
2,0.43
3,0.43
4,0.39
5,0.34
6,0.25
7,0.33


Databricks visualization. Run in Databricks to view.

In [0]:
display(gold_age_attrition_rate)

Age,Attrition_Rate
22,0.45
23,0.32
24,0.5
25,0.53
26,0.53
27,0.47
28,0.39
29,0.36
30,0.4
31,0.3


Databricks visualization. Run in Databricks to view.

In [0]:
display(gold_discount_effect)

Discount,Avg_Sales
0.0,4511.5
0.01,5103.34
0.02,4435.17
0.03,5604.86
0.04,5234.55
0.05,5741.82
0.06,4432.35
0.07,4599.94
0.08,4967.66
0.09,4794.34


Databricks visualization. Run in Databricks to view.

In [0]:
display(gold_quantity_revenue)

Quantity_Sold,Total_Revenue
1,96007.87
2,108691.16
3,73111.32
4,114444.85
5,69266.69
6,94913.11
7,90102.79
8,86281.12
9,84207.78
10,87890.39


Databricks visualization. Run in Databricks to view.